# 3. PyCaret Modeling (Polars Optimized)

This version uses Polars for fast data loading/sampling and keeps PyCaret for model training.

Performance knobs are included to avoid long-running tuning/stacking on large datasets.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl

from pycaret.classification import (
    setup,
    compare_models,
    tune_model,
    blend_models,
    stack_models,
    finalize_model,
    predict_model,
    pull,
    save_model,
)
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks_polars':
    ROOT = ROOT.parent

PROCESSED = ROOT / 'data' / 'processed'
POWERBI_OUT = PROCESSED / 'powerbi'
MODELS_OUT = ROOT / 'models'
ML_OUT = PROCESSED / 'ml'

MODELS_OUT.mkdir(parents=True, exist_ok=True)
ML_OUT.mkdir(parents=True, exist_ok=True)
POWERBI_OUT.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)

ROOT: C:\Users\sayee\Documents\Credit Default Risk


In [10]:
TRAIN_SAMPLE_SIZE = 10000
TOP_N_MODELS = 3
TUNE_TOP_N = 1

SKIP_TUNING = False
FAST_TUNE_ITER = 12

ENABLE_BLENDING = False
ENABLE_STACKING = False

PRIMARY_SORT_METRIC = 'Recall'
TUNE_OPTIMIZE_METRIC = 'AUC'
PYCARET_N_JOBS = 1

CANDIDATE_MODEL_IDS = ['ada', 'lr', 'ridge']

print('Config loaded')

Config loaded


In [11]:
train_path = PROCESSED / 'train_features.parquet'
test_path = PROCESSED / 'test_features.parquet'

train_pl = pl.read_parquet(train_path)
test_pl = pl.read_parquet(test_path)

train_pl = train_pl.with_columns(pl.col('TARGET').cast(pl.Int32))

sample_size = min(TRAIN_SAMPLE_SIZE, train_pl.height)
train_sample_pl = train_pl.sample(n=sample_size, seed=42, shuffle=True)

# Use standard pandas dtypes to avoid pd.NA ambiguity in sklearn/PyCaret.
train_sample = train_sample_pl.to_pandas()
train_df = train_pl.to_pandas()
test_df = test_pl.to_pandas()

train_sample = train_sample.replace({pd.NA: np.nan})
train_df = train_df.replace({pd.NA: np.nan})
test_df = test_df.replace({pd.NA: np.nan})

print(f'Train full shape: {train_df.shape}')
print(f'Train sample shape for PyCaret compare/tune: {train_sample.shape}')
print(f'Test shape: {test_df.shape}')

Train full shape: (307511, 388)
Train sample shape for PyCaret compare/tune: (40000, 388)
Test shape: (48744, 387)


In [4]:
def score_column(df: pd.DataFrame) -> str:
    preferred = [
        'prediction_score',
        'prediction_score_1',
        'Score',
        'Score_1',
        'score',
        'score_1',
    ]
    for c in preferred:
        if c in df.columns:
            return c

    score_like = [
        c for c in df.columns
        if 'score' in c.lower() or 'prob' in c.lower()
    ]
    if score_like:
        return score_like[0]

    raise KeyError(f'No score/probability column found. Available columns: {list(df.columns)}')


def label_column(df: pd.DataFrame) -> str:
    for c in ['prediction_label', 'Label', 'label', 'prediction']:
        if c in df.columns:
            return c
    raise KeyError(f'No label column found. Available columns: {list(df.columns)}')


def orient_positive_class_scores(scores: pd.Series, target: pd.Series) -> tuple[pd.Series, bool]:
    scores = pd.to_numeric(scores, errors='coerce').astype(float)
    bad_mean = scores[target == 1].mean()
    good_mean = scores[target == 0].mean()
    invert_scores = pd.notna(bad_mean) and pd.notna(good_mean) and bad_mean < good_mean
    if invert_scores:
        scores = 1.0 - scores
    return scores.clip(0.0, 1.0), invert_scores

In [12]:
exp = setup(
    data=train_sample,
    target='TARGET',
    session_id=42,
    fold=3,
    train_size=0.8,
    preprocess=True,
    imputation_type='simple',
    numeric_imputation='median',
    categorical_imputation='most_frequent',
    fix_imbalance=False,
    remove_multicollinearity=False,
    n_jobs=PYCARET_N_JOBS,
    verbose=False,
)

top_models = compare_models(
    sort=PRIMARY_SORT_METRIC,
    n_select=TOP_N_MODELS,
    turbo=True,
    include=CANDIDATE_MODEL_IDS,
    errors='ignore',
)
leaderboard = pull().copy()
if 'AUC' in leaderboard.columns and leaderboard['AUC'].fillna(0).eq(0).all():
    print('Warning: compare_models returned zero AUC values. Using recall-first ranking for candidate selection.')
leaderboard.to_csv(ML_OUT / 'model_leaderboard_polars.csv', index=False)
leaderboard.to_parquet(ML_OUT / 'model_leaderboard_polars.parquet', index=False)

if not isinstance(top_models, list):
    top_models = [top_models]
top_models = [m for m in top_models if m is not None]
if not top_models:
    raise RuntimeError('compare_models did not return any valid models. Check the PyCaret environment and candidate model list.')

print('Leaderboard rows:', len(leaderboard))
display(leaderboard.head(15))

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.9167,0.0000,0.0008,0.5000,0.0015,0.0012,0.0166,4.2700
et,Extra Trees Classifier,0.9167,0.0000,0.0000,0.0000,0.0000,-0.0001,-0.0010,2.8300
gbc,Gradient Boosting Classifier,0.9168,0.0000,0.0281,0.5196,0.0533,0.0450,0.1059,25.5933
ada,Ada Boost Classifier,0.9154,0.0000,0.0473,0.4313,0.0850,0.0697,0.1205,5.8467
lr,Logistic Regression,0.9157,0.0000,0.0011,0.0750,0.0022,-0.0001,-0.0008,4.3867
ridge,Ridge Classifier,0.9164,0.0000,0.0034,0.3742,0.0067,0.0050,0.0276,1.0300


Leaderboard rows: 6


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.9167,0.0,0.0008,0.5000,0.0015,0.0012,0.0166,4.2700
et,Extra Trees Classifier,0.9167,0.0,0.0000,0.0000,0.0000,-0.0001,-0.0010,2.8300
gbc,Gradient Boosting Classifier,0.9168,0.0,0.0281,0.5196,0.0533,0.0450,0.1059,25.5933
ada,Ada Boost Classifier,0.9154,0.0,0.0473,0.4313,0.0850,0.0697,0.1205,5.8467
lr,Logistic Regression,0.9157,0.0,0.0011,0.0750,0.0022,-0.0001,-0.0008,4.3867
ridge,Ridge Classifier,0.9164,0.0,0.0034,0.3742,0.0067,0.0050,0.0276,1.0300


In [13]:
if SKIP_TUNING:
    tuned_models = list(top_models)
else:
    tuned_models = []
    for m in top_models[:TUNE_TOP_N]:
        try:
            tuned = tune_model(
                m,
                optimize=TUNE_OPTIMIZE_METRIC,
                choose_better=True,
                n_iter=FAST_TUNE_ITER,
                verbose=False,
            )
            tuned_models.append(tuned)
        except Exception:
            tuned_models.append(m)
    for m in top_models[TUNE_TOP_N:]:
        tuned_models.append(m)

candidate_models = tuned_models.copy()

if ENABLE_BLENDING and len(tuned_models) >= 2:
    try:
        blended = blend_models(estimator_list=tuned_models, optimize=TUNE_OPTIMIZE_METRIC, verbose=False)
        candidate_models.append(blended)
    except Exception:
        pass

if ENABLE_STACKING and len(tuned_models) >= 2:
    try:
        stacked = stack_models(estimator_list=tuned_models, optimize=TUNE_OPTIMIZE_METRIC, verbose=False)
        candidate_models.append(stacked)
    except Exception:
        pass

metrics_rows = []
best_model = None
best_candidate_key = (-1.0, -1.0)

for i, model in enumerate(candidate_models, start=1):
    _ = predict_model(model)
    m = pull().copy()
    m['candidate_index'] = i
    metrics_rows.append(m)

    recall = float(m['Recall'].iloc[0]) if 'Recall' in m.columns else np.nan
    auc = float(m['AUC'].iloc[0]) if 'AUC' in m.columns else np.nan
    candidate_key = (
        np.nan_to_num(recall, nan=-1.0),
        np.nan_to_num(auc, nan=-1.0),
    )
    if candidate_key > best_candidate_key:
        best_candidate_key = candidate_key
        best_model = model

metrics_df = pd.concat(metrics_rows, ignore_index=True) if metrics_rows else pd.DataFrame()
metrics_df.to_csv(ML_OUT / 'model_holdout_metrics_polars.csv', index=False)
if not metrics_df.empty:
    metrics_df.to_parquet(ML_OUT / 'model_holdout_metrics_polars.parquet', index=False)

if best_model is None and candidate_models:
    best_model = candidate_models[0]

display(metrics_df)
print(
    f'SKIP_TUNING={SKIP_TUNING} | TUNE_TOP_N={TUNE_TOP_N} '
    f'| Best holdout Recall={best_candidate_key[0]:.6f} | Best holdout AUC={best_candidate_key[1]:.6f}'
)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,0.9168,0.6899,0.0015,1.0000,0.0030,0.0027,0.0371


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Extra Trees Classifier,0.9166,0.7096,0.0015,0.5000,0.0030,0.0025,0.0238


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Gradient Boosting Classifier,0.9171,0.7495,0.0345,0.5476,0.0649,0.0556,0.1220


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Ada Boost Classifier,0.9156,0.7320,0.0570,0.4524,0.1012,0.0841,0.1375


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,candidate_index
0,Random Forest Classifier,0.9168,0.6899,0.0015,1.0000,0.0030,0.0027,0.0371,1
1,Extra Trees Classifier,0.9166,0.7096,0.0015,0.5000,0.0030,0.0025,0.0238,2
2,Gradient Boosting Classifier,0.9171,0.7495,0.0345,0.5476,0.0649,0.0556,0.1220,3
3,Ada Boost Classifier,0.9156,0.7320,0.0570,0.4524,0.1012,0.0841,0.1375,4


SKIP_TUNING=False | TUNE_TOP_N=1 | Best holdout AUC: 0.749500


In [14]:
final_model = finalize_model(best_model)
save_model(final_model, str(MODELS_OUT / 'best_pycaret_model_polars'))

train_scored = predict_model(final_model, data=train_df.drop(columns=['TARGET']))
test_scored = predict_model(final_model, data=test_df)

train_scored['TARGET'] = train_df['TARGET'].values

score_col_train = score_column(train_scored)
label_col_train = label_column(train_scored)
score_col_test = score_column(test_scored)
label_col_test = label_column(test_scored)

train_scores, invert_scores = orient_positive_class_scores(train_scored[score_col_train], train_df['TARGET'])
test_scores = pd.to_numeric(test_scored[score_col_test], errors='coerce').astype(float)
if invert_scores:
    test_scores = 1.0 - test_scores
test_scores = test_scores.clip(0.0, 1.0)

train_pred_out = pd.DataFrame(
    {
        'SK_ID_CURR': train_df['SK_ID_CURR'].values,
        'DATASET': 'train',
        'TARGET': train_scored['TARGET'].values,
        'PRED_LABEL': train_scored[label_col_train].values,
        'PRED_SCORE': train_scores.values,
    }
)

test_pred_out = pd.DataFrame(
    {
        'SK_ID_CURR': test_df['SK_ID_CURR'].values,
        'DATASET': 'test',
        'TARGET': np.nan,
        'PRED_LABEL': test_scored[label_col_test].values,
        'PRED_SCORE': test_scores.values,
    }
)

model_scores = pd.concat([train_pred_out, test_pred_out], ignore_index=True)
model_scores['PRED_RISK_BAND'] = pd.qcut(
    model_scores['PRED_SCORE'].fillna(model_scores['PRED_SCORE'].median()),
    q=5,
    labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'],
    duplicates='drop',
)

train_auc = roc_auc_score(train_df['TARGET'], train_pred_out['PRED_SCORE'])
train_band_default_rate = train_pred_out.assign(
    PRED_RISK_BAND=model_scores.loc[model_scores['DATASET'] == 'train', 'PRED_RISK_BAND'].values
).groupby('PRED_RISK_BAND', observed=False)['TARGET'].mean()

train_pred_out.to_parquet(ML_OUT / 'train_predictions_polars.parquet', index=False)
train_pred_out.to_csv(ML_OUT / 'train_predictions_polars.csv', index=False)
test_pred_out.to_parquet(ML_OUT / 'test_predictions_polars.parquet', index=False)
test_pred_out.to_csv(ML_OUT / 'test_predictions_polars.csv', index=False)

model_scores.to_parquet(POWERBI_OUT / 'fact_model_scores_polars.parquet', index=False)
model_scores.to_csv(POWERBI_OUT / 'fact_model_scores_polars.csv', index=False)

print('Polars-based PyCaret training and scoring complete')
print('Model scores rows:', len(model_scores))
print(f'Score column used: {score_col_train} | Inverted to positive-class risk: {invert_scores}')
print(f'Train ROC AUC using exported scores: {train_auc:.6f}')
print('Train default rate by predicted risk band:')
print(train_band_default_rate)
model_scores.head()

Transformation Pipeline and Model Successfully Saved


Polars-based PyCaret training and scoring complete
Model scores rows: 356255


,SK_ID_CURR,DATASET,TARGET,PRED_LABEL,PRED_SCORE,PRED_RISK_BAND
0,100002,train,1.0,0,0.5056,Very Low
1,100003,train,0.0,0,0.9753,Very High
2,100004,train,0.0,0,0.9530,Medium
3,100006,train,0.0,0,0.9525,Medium
4,100007,train,0.0,0,0.9220,Low
